# RDS Full Load Validation
Gabriel Ferreira

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv

### 2. Starting Spark Session

In [2]:
spark = SparkSession.builder \
    .appName("FraudDetectionLakehouse") \
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
    ) \
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    ) \
    .getOrCreate()

print(spark)

### 3. AWS S3 Connection

In [3]:
# Configuring AWS credentials

load_dotenv("/home/jovyan/work/.env")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.access.key",
    AWS_ACCESS_KEY
)

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.secret.key",
    AWS_SECRET_KEY
)

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.endpoint",
    "s3.amazonaws.com"
)

### RDS Validation -- after run FULL LOAD

In [4]:
conn = psycopg2.connect(
    host="fraud-detection-rds.cilk4oyi2c5j.us-east-1.rds.amazonaws.com",
    port="5432",
    database="fraud_analytics",
    user="postgres",
    password="fraud-gabriel-rds-2026"
)

print("Connected successfully!")

Connected successfully!


In [5]:
# Validar fraud_predictions
query = """
SELECT COUNT(*)
FROM fraud_transactions_full;
"""

pd.read_sql(query, conn)

/tmp/ipykernel_11005/1908251729.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,count
0,1316675


In [6]:
conn.close()

print("Connection closed successfully!")

Connection closed successfully!
